In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, f1_score
import warnings


In [4]:
import pandas as pd
df = pd.read_csv('women-clothing-accessories.3-class.balanced.csv', sep='\t')
df.head()

,review,sentiment
0,качество плохое пошив ужасный (горловина напер...,negative
1,"Товар отдали другому человеку, я не получила п...",negative
2,"Ужасная синтетика! Тонкая, ничего общего с пре...",negative
3,"товар не пришел, продавец продлил защиту без м...",negative
4,"Кофточка голая синтетика, носить не возможно.",negative


In [5]:
print(f"Размер датасета: {df.shape}")
print(f"Колонки: {df.columns.tolist()}")
print(f"Распределение классов:\n{df['sentiment'].value_counts()}")

Размер датасета: (90000, 2)
Колонки: ['review', 'sentiment']
Распределение классов:
sentiment
negative    30000
neautral    30000
positive    30000
Name: count, dtype: int64


In [6]:
# Приводим метки к числовому формату
label_map = {'negative': 0, 'neautral': 1, 'positive': 2}
df['label'] = df['sentiment'].map(label_map)

# Для скорости можно взять subsample (на полном датасете будет лучше)
# df = df.sample(30000, random_state=42)   # раскомментируйте при необходимости

X_train, X_test, y_train, y_test = train_test_split(
    df['review'], df['label'], 
    test_size=0.2, 
    stratify=df['label'], 
    random_state=42
)

# TF-IDF векторизация
vectorizer = TfidfVectorizer(
    max_features=15000, 
    ngram_range=(1, 2),
    min_df=2
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [8]:
# 1. Logistic Regression 
lr = LogisticRegression(
    max_iter=1000, 
    solver='lbfgs',      
    n_jobs=-1
)
lr.fit(X_train_vec, y_train)
y_pred_lr = lr.predict(X_test_vec)

# 2. Random Forest
rf = RandomForestClassifier(
    n_estimators=300, 
    max_depth=None, 
    random_state=42, 
    n_jobs=-1
)
rf.fit(X_train_vec, y_train)
y_pred_rf = rf.predict(X_test_vec)

# 3. Linear SVC
svc = LinearSVC(
    max_iter=2000, 
    dual=False, 
    random_state=42
)
svc.fit(X_train_vec, y_train)
y_pred_svc = svc.predict(X_test_vec)

In [9]:
def evaluate_model(y_true, y_pred, model_name):
    print(f"\n=== {model_name} ===")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"F1-macro:  {f1_score(y_true, y_pred, average='macro'):.4f}")
    print(classification_report(y_true, y_pred, 
                                target_names=['negative', 'neutral', 'positive'],
                                digits=4))

evaluate_model(y_test, y_pred_lr,  "Logistic Regression")
evaluate_model(y_test, y_pred_rf,  "Random Forest")
evaluate_model(y_test, y_pred_svc, "Linear SVC")


=== Logistic Regression ===
Accuracy:  0.7528
F1-macro:  0.7542
              precision    recall  f1-score   support

    negative     0.7408    0.7092    0.7246      6000
     neutral     0.6393    0.6830    0.6604      6000
    positive     0.8892    0.8663    0.8776      6000

    accuracy                         0.7528     18000
   macro avg     0.7564    0.7528    0.7542     18000
weighted avg     0.7564    0.7528    0.7542     18000


=== Random Forest ===
Accuracy:  0.7302
F1-macro:  0.7308
              precision    recall  f1-score   support

    negative     0.7452    0.6788    0.7104      6000
     neutral     0.6166    0.6630    0.6389      6000
    positive     0.8372    0.8487    0.8429      6000

    accuracy                         0.7302     18000
   macro avg     0.7330    0.7302    0.7308     18000
weighted avg     0.7330    0.7302    0.7308     18000


=== Linear SVC ===
Accuracy:  0.7350
F1-macro:  0.7354
              precision    recall  f1-score   support

   

In [10]:
#!pip install transformers torch
from transformers import pipeline
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Загрузка данных
df = pd.read_csv('women-clothing-accessories.3-class.balanced.csv', sep='\t')

label_map = {'negative': 0, 'neautral': 1, 'positive': 2}
df['label'] = df['sentiment'].map(label_map)

# Для скорости берём тестовую выборку 
test_df = df.sample(5000, random_state=42)   

X_test = test_df['review'].tolist()
y_test = test_df['label'].values

# Загрузка модели (RuBERT sentiment)
print("Загружаем модель RuBERT для русского языка...")
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="seara/rubert-tiny2-russian-sentiment",
    tokenizer="seara/rubert-tiny2-russian-sentiment",
    return_all_scores=False,
    device=-1   # -1 = CPU, 0 = GPU (если доступен)
)

# Предсказание 
print("Выполняем предсказание...")
predictions = []

for i, text in enumerate(X_test):
    if i % 500 == 0:
        print(f"Обработано {i}/{len(X_test)} отзывов...")
    
    result = sentiment_pipeline(text[:512])[0]   
    label = result['label']
    
    # Маппинг меток модели на наши классы
    if label == 'negative':
        pred = 0
    elif label == 'positive':
        pred = 2
    else:  # neutral
        pred = 1
    predictions.append(pred)

y_pred_bert = np.array(predictions)

# выводим результаты =

print("РЕЗУЛЬТАТЫ RuBERT (seara/rubert-tiny2-russian-sentiment)")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_bert):.4f}")
print(f"F1-macro:  {f1_score(y_test, y_pred_bert, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_bert, 
                           target_names=['negative', 'neutral', 'positive'],
                           digits=4))

Загружаем модель RuBERT для русского языка...


Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: seara/rubert-tiny2-russian-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Выполняем предсказание...
Обработано 0/5000 отзывов...
Обработано 500/5000 отзывов...
Обработано 1000/5000 отзывов...
Обработано 1500/5000 отзывов...
Обработано 2000/5000 отзывов...
Обработано 2500/5000 отзывов...
Обработано 3000/5000 отзывов...
Обработано 3500/5000 отзывов...
Обработано 4000/5000 отзывов...
Обработано 4500/5000 отзывов...
РЕЗУЛЬТАТЫ RuBERT (seara/rubert-tiny2-russian-sentiment)
Accuracy:  0.7614
F1-macro:  0.7647

Classification Report:
              precision    recall  f1-score   support

    negative     0.7636    0.6996    0.7302      1671
     neutral     0.6457    0.7194    0.6806      1700
    positive     0.8984    0.8686    0.8833      1629

    accuracy                         0.7614      5000
   macro avg     0.7692    0.7625    0.7647      5000
weighted avg     0.7674    0.7614    0.7632      5000



In [ ]:
#Итоговые выводы и рекомендации
#Победитель среди протестированных моделей — Linear SVC , она даёт наилучшее сочетание качества и скорости.
#Классические ML-модели на этом датасете пока превосходят лёгкую RuBERT-tiny2, так как датасет довольно специфический (короткие русские отзывы с большим количеством сленга, опечаток и эмоциональной лексики), Классические модели с качественным TF-IDF + n-граммами очень хорошо справляются с такой задачей.